In [ ]:
!apt-get update -qq
!apt-get install -y r-base


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
r-base is already the newest version (4.5.2-1.2204.0).
0 upgraded, 0 newly installed, 0 to remove and 98 not upgraded.


In [ ]:
!R -e 'install.packages("synthpop", repos="https://cloud.r-project.org")'


Streaming output truncated to the last 5000 lines.
../inst/include/Eigen/src/Core/CoreEvaluators.h:100:8:   required from ‘struct Eigen::internal::evaluator<const Eigen::Block<const Eigen::Block<Eigen::Block<Eigen::Block<Eigen::Matrix<double, -1, -1>, -1, -1, false>, -1, 1, true>, -1, 1, false>, -1, 1, false> >’
../inst/include/Eigen/src/Core/CoreEvaluators.h:564:45:   required from ‘struct Eigen::internal::unary_evaluator<Eigen::CwiseUnaryOp<Eigen::internal::scalar_abs2_op<double>, const Eigen::Block<const Eigen::Block<Eigen::Block<Eigen::Block<Eigen::Matrix<double, -1, -1>, -1, -1, false>, -1, 1, true>, -1, 1, false>, -1, 1, false> >, Eigen::internal::IndexBased, double>’
../inst/include/Eigen/src/Core/CoreEvaluators.h:90:8:   required from ‘struct Eigen::internal::evaluator<Eigen::CwiseUnaryOp<Eigen::internal::scalar_abs2_op<double>, const Eigen::Block<const Eigen::Block<Eigen::Block<Eigen::Block<Eigen::Matrix<double, -1, -1>, -1, -1, false>, -1, 1, true>, -1, 1, false>, -1, 1, fals

In [3]:
!R -e 'library(synthpop); cat("SynthPop loaded successfully\n")'



R version 4.5.2 (2025-10-31) -- "[Not] Part in a Rumble"
Copyright (C) 2025 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> library(synthpop); cat("SynthPop loaded successfully\n")
New version of synthpop (1.9-0) with disclosure functions
see disclosure.pdf for details and NEWS file for other changes

Find out more at https://www.synthpop.org.uk/
SynthPop loaded successfully
> 


In [5]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/katabatic1

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/katabatic1


In [ ]:
from pathlib import Path
import pandas as pd
import time
import subprocess

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "adult.csv"
DATASET_DIR = ROOT / "sample_data" / "adult"
SYNTH_DIR = ROOT / "synthetic" / "adult" / "synthpop"

DATASET_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
pipeline_start = time.time()

from katabatic.utils.split_dataset import split_dataset

print("Splitting ADULT dataset")

split_dataset(
    input_csv=str(RAW_CSV),
    output_dir=str(DATASET_DIR),
    label_col="income",
    test_size=0.2,
    stratify=True,
    random_state=42
)

print("Split complete")

print("Running SynthPop (ADULT)")

x_train = pd.read_csv(DATASET_DIR / "x_train.csv")
y_train = pd.read_csv(DATASET_DIR / "y_train.csv")

label_col = y_train.columns[0]

train_full = pd.concat([x_train, y_train], axis=1)
train_csv = SYNTH_DIR / "train_full.csv"
train_full.to_csv(train_csv, index=False)

r_script = f"""
suppressMessages(library(synthpop))
set.seed(42)

data <- read.csv("{train_csv.as_posix()}")

syn_data <- syn(
  data,
  method = "cart",
  seed = 42
)

write.csv(
  syn_data$syn,
  "{(SYNTH_DIR / 'synthetic_full.csv').as_posix()}",
  row.names = FALSE
)
"""

r_path = SYNTH_DIR / "run_synthpop.R"
r_path.write_text(r_script.strip())

subprocess.run(
    ["Rscript", str(r_path)],
    check=True
)

print("SynthPop generation complete")

synth_full = pd.read_csv(SYNTH_DIR / "synthetic_full.csv")

def normalize_columns(df):
    df.columns = (
        df.columns
        .str.replace(".", "-", regex=False)
        .str.strip()
    )
    return df

synth_full = normalize_columns(synth_full)

x_synth = synth_full.drop(columns=[label_col])
y_synth = synth_full[[label_col]]

x_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Synthetic X and y saved")

from sklearn.preprocessing import OrdinalEncoder

print("Encoding features for TSTR")

x_test = pd.read_csv(DATASET_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

x_test = normalize_columns(x_test)
x_synth = normalize_columns(x_synth)

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_test_enc = encoder.fit_transform(x_test.astype(str))
x_synth_enc = encoder.transform(x_synth.astype(str))

pd.DataFrame(x_test_enc, columns=x_test.columns).to_csv(
    DATASET_DIR / "x_test.csv", index=False
)

pd.DataFrame(x_synth_enc, columns=x_synth.columns).to_csv(
    SYNTH_DIR / "x_synth.csv", index=False
)

print("Feature encoding complete")

print("Reindexing labels")

y_train = pd.read_csv(DATASET_DIR / "y_train.csv")
y_test = pd.read_csv(DATASET_DIR / "y_test.csv")
y_synth = pd.read_csv(SYNTH_DIR / "y_synth.csv")

label_col = y_train.columns[0]

all_labels = pd.concat([y_train, y_test, y_synth])[label_col].unique()
all_labels = sorted(all_labels)

label_map = {old: new for new, old in enumerate(all_labels)}
print("Label mapping:", label_map)

for df in [y_train, y_test, y_synth]:
    df[label_col] = df[label_col].map(label_map)

y_train.to_csv(DATASET_DIR / "y_train.csv", index=False)
y_test.to_csv(DATASET_DIR / "y_test.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Label reindexing complete")

from katabatic.evaluate.tstr.evaluation import TSTREvaluation

print("Running TSTR evaluation")

tstr_start = time.time()

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(DATASET_DIR)
)

results = tstr.evaluate()

tstr_end = time.time()
print(f"TSTR completed in {(tstr_end - tstr_start)/60:.2f} minutes")

print("TSTR Results")
print(results)

pipeline_end = time.time()
print(f"Total pipeline runtime: {(pipeline_end - pipeline_start)/60:.2f} minutes")

print("SYNTHPOP (ADULT) PIPELINE FINISHED SUCCESSFULLY")


ROOT: /content/drive/MyDrive/katabatic1

▶ Splitting ADULT dataset
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
<=50K    0.759175
>50K     0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
<=50K    0.759251
>50K     0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
✔ Split complete

▶ Running SynthPop (ADULT)
✔ SynthPop generation complete
✔ Synthetic X / y saved

▶ Encoding features for TSTR
✔ Feature encoding complete

▶ Reindexing labels
Label mapping: {' <=50K': 0, ' >50K': 1}
✔ Label reindexing complete

▶ Running TSTR evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/adult/synthpop_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7878
F1 Score: 0.7615
AUC: 0.8145

MLP:
Accuracy: 0.8411
F1 Score: 0.8390
AUC: 0.8908

RF:
Accuracy: 0.8485
F1 Score: 0.8451
AUC: 0.8961

XGBoost:
Accuracy: 0.8305
F1 Score: 0.8371
AUC: 0.9098
✔ TSTR completed in 0.34 minutes

📊 TSTR RESULTS
{'LR': {'Accuracy': 0.7878089973898357, 'F1 Score': 0.76154312731757, 'AUC': np.float64(0.8144760735436742)}, 'MLP': {'Accuracy': 0.841087056655919, 'F1 Score': 0.838975690631281, 'AUC': np.float64(0.8908176162274819)}, 'RF': {'Accuracy': 0.8484569322892677, 'F1 Score': 0.8450981066960813, 'AUC': np.float64(0.8960626199418089)}, 'XGBoost': {'Accuracy': 0.8304928604329802, 'F1 Score': 0.8370882575777351, 'AUC': np.float64(0.9098128520872454)}}

⏱️ Total pipeline runtime: 1.02 minutes

✅ SYNTHPOP (ADULT) PIPELINE FINISHED SUCCESSFULLY


In [ ]:
from pathlib import Path
import pandas as pd
import time
import subprocess

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "shuttle.csv"
DATASET_DIR = ROOT / "sample_data" / "shuttle"
SYNTH_DIR = ROOT / "synthetic" / "shuttle" / "synthpop"

DATASET_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
pipeline_start = time.time()

from katabatic.utils.split_dataset import split_dataset

print("Splitting SHUTTLE dataset")

split_dataset(
    input_csv=str(RAW_CSV),
    output_dir=str(DATASET_DIR),
    label_col="target",
    test_size=0.2,
    stratify=True,
    random_state=42
)

print("Split complete")

print("Running SynthPop (SHUTTLE)")

x_train = pd.read_csv(DATASET_DIR / "x_train.csv")
y_train = pd.read_csv(DATASET_DIR / "y_train.csv")

label_col = y_train.columns[0]

train_full = pd.concat([x_train, y_train], axis=1)
train_csv = SYNTH_DIR / "train_full.csv"
train_full.to_csv(train_csv, index=False)

r_script = f"""
suppressMessages(library(synthpop))
set.seed(42)

data <- read.csv("{train_csv.as_posix()}")

syn_data <- syn(
  data,
  method = "cart",
  seed = 42
)

write.csv(
  syn_data$syn,
  "{(SYNTH_DIR / 'synthetic_full.csv').as_posix()}",
  row.names = FALSE
)
"""

r_path = SYNTH_DIR / "run_synthpop.R"
r_path.write_text(r_script.strip())

subprocess.run(
    ["Rscript", str(r_path)],
    check=True
)

print("SynthPop generation complete")

synth_full = pd.read_csv(SYNTH_DIR / "synthetic_full.csv")

def normalize_columns(df):
    df.columns = (
        df.columns
        .str.replace(".", "-", regex=False)
        .str.strip()
    )
    return df

synth_full = normalize_columns(synth_full)

x_synth = synth_full.drop(columns=[label_col])
y_synth = synth_full[[label_col]]

x_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Synthetic X and y saved")

from sklearn.preprocessing import OrdinalEncoder

print("Encoding features for TSTR")

x_test = pd.read_csv(DATASET_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

x_test = normalize_columns(x_test)
x_synth = normalize_columns(x_synth)

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_test_enc = encoder.fit_transform(x_test.astype(str))
x_synth_enc = encoder.transform(x_synth.astype(str))

pd.DataFrame(x_test_enc, columns=x_test.columns).to_csv(
    DATASET_DIR / "x_test.csv", index=False
)

pd.DataFrame(x_synth_enc, columns=x_synth.columns).to_csv(
    SYNTH_DIR / "x_synth.csv", index=False
)

print("Feature encoding complete")

print("Reindexing labels")

y_train = pd.read_csv(DATASET_DIR / "y_train.csv")
y_test = pd.read_csv(DATASET_DIR / "y_test.csv")
y_synth = pd.read_csv(SYNTH_DIR / "y_synth.csv")

label_col = y_train.columns[0]

all_labels = pd.concat([y_train, y_test, y_synth])[label_col].unique()
all_labels = sorted(all_labels)

label_map = {old: new for new, old in enumerate(all_labels)}
print("Label mapping:", label_map)

for df in [y_train, y_test, y_synth]:
    df[label_col] = df[label_col].map(label_map)

y_train.to_csv(DATASET_DIR / "y_train.csv", index=False)
y_test.to_csv(DATASET_DIR / "y_test.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Label reindexing complete")

from katabatic.evaluate.tstr.evaluation import TSTREvaluation

print("Running TSTR evaluation")

tstr_start = time.time()

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(DATASET_DIR)
)

results = tstr.evaluate()

tstr_end = time.time()
print(f"TSTR completed in {(tstr_end - tstr_start)/60:.2f} minutes")

print("TSTR Results")
print(results)

pipeline_end = time.time()
print(f"Total pipeline runtime: {(pipeline_end - pipeline_start)/60:.2f} minutes")

print("SYNTHPOP (SHUTTLE) PIPELINE FINISHED SUCCESSFULLY")


ROOT: /content/drive/MyDrive/katabatic1

▶ Splitting SHUTTLE dataset
Loaded data with shape: (58000, 10)
Saved train/test full data
Train size: (46400, 10), Test size: (11600, 10)
Train label distribution:
 class
1    0.785970
4    0.153491
5    0.056336
3    0.002953
2    0.000862
7    0.000216
6    0.000172
Name: proportion, dtype: float64
Test label distribution:
 class
1    0.785948
4    0.153534
5    0.056293
3    0.002931
2    0.000862
7    0.000259
6    0.000172
Name: proportion, dtype: float64
Saved X/y split
Training shape: (46400, 9) (46400,)
Test shape: (11600, 9) (11600,)
✔ Split complete

▶ Running SynthPop (SHUTTLE)
✔ SynthPop generation complete
✔ Synthetic X / y saved

▶ Encoding features for TSTR
✔ Feature encoding complete

▶ Reindexing labels
Label mapping: {np.int64(1): 0, np.int64(2): 1, np.int64(3): 2, np.int64(4): 3, np.int64(5): 4, np.int64(6): 5, np.int64(7): 6}
✔ Label reindexing complete

▶ Running TSTR evaluation


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [01:25:31] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/shuttle/synthpop_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.9581
F1 Score: 0.9573

MLP:
Accuracy: 0.9988
F1 Score: 0.9987

RF:
Accuracy: 0.9990
F1 Score: 0.9989

XGBoost:
Accuracy: 0.9991
F1 Score: 0.9991
✔ TSTR completed in 0.28 minutes

📊 TSTR RESULTS
{'LR': {'Accuracy': 0.958103448275862, 'F1 Score': 0.9572971227948773}, 'MLP': {'Accuracy': 0.9987931034482759, 'F1 Score': 0.9987045698926719}, 'RF': {'Accuracy': 0.9989655172413793, 'F1 Score': 0.9988739006954195}, 'XGBoost': {'Accuracy': 0.9991379310344828, 'F1 Score': 0.9990513187106522}}

⏱️ Total pipeline runtime: 0.79 minutes

✅ SYNTHPOP (SHUTTLE) PIPELINE FINISHED SUCCESSFULLY


In [ ]:
from pathlib import Path
import pandas as pd
import time
import subprocess

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "magic.csv"
DATASET_DIR = ROOT / "sample_data" / "magic"
SYNTH_DIR = ROOT / "synthetic" / "magic" / "synthpop"

DATASET_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
pipeline_start = time.time()

from katabatic.utils.split_dataset import split_dataset

print("Splitting MAGIC dataset")

split_dataset(
    input_csv=str(RAW_CSV),
    output_dir=str(DATASET_DIR),
    label_col="class",
    test_size=0.2,
    stratify=True,
    random_state=42
)

print("Split complete")

print("Running SynthPop (MAGIC)")

x_train = pd.read_csv(DATASET_DIR / "x_train.csv")
y_train = pd.read_csv(DATASET_DIR / "y_train.csv")

label_col = y_train.columns[0]

train_full = pd.concat([x_train, y_train], axis=1)
train_csv = SYNTH_DIR / "train_full.csv"
train_full.to_csv(train_csv, index=False)

r_script = f"""
suppressMessages(library(synthpop))
set.seed(42)

data <- read.csv("{train_csv.as_posix()}")

syn_data <- syn(
  data,
  method = "cart",
  seed = 42
)

write.csv(
  syn_data$syn,
  "{(SYNTH_DIR / 'synthetic_full.csv').as_posix()}",
  row.names = FALSE
)
"""

r_path = SYNTH_DIR / "run_synthpop.R"
r_path.write_text(r_script.strip())

subprocess.run(
    ["Rscript", str(r_path)],
    check=True
)

print("SynthPop generation complete")

synth_full = pd.read_csv(SYNTH_DIR / "synthetic_full.csv")

def normalize_columns(df):
    df.columns = (
        df.columns
        .str.replace(".", "-", regex=False)
        .str.strip()
    )
    return df

synth_full = normalize_columns(synth_full)

x_synth = synth_full.drop(columns=[label_col])
y_synth = synth_full[[label_col]]

x_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Synthetic X and y saved")

from sklearn.preprocessing import OrdinalEncoder

print("Encoding features for TSTR")

x_test = pd.read_csv(DATASET_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

x_test = normalize_columns(x_test)
x_synth = normalize_columns(x_synth)

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_test_enc = encoder.fit_transform(x_test.astype(str))
x_synth_enc = encoder.transform(x_synth.astype(str))

pd.DataFrame(x_test_enc, columns=x_test.columns).to_csv(
    DATASET_DIR / "x_test.csv", index=False
)

pd.DataFrame(x_synth_enc, columns=x_synth.columns).to_csv(
    SYNTH_DIR / "x_synth.csv", index=False
)

print("Feature encoding complete")

print("Reindexing labels")

y_train = pd.read_csv(DATASET_DIR / "y_train.csv")
y_test = pd.read_csv(DATASET_DIR / "y_test.csv")
y_synth = pd.read_csv(SYNTH_DIR / "y_synth.csv")

label_col = y_train.columns[0]

all_labels = pd.concat([y_train, y_test, y_synth])[label_col].unique()
all_labels = sorted(all_labels)

label_map = {old: new for new, old in enumerate(all_labels)}
print("Label mapping:", label_map)

for df in [y_train, y_test, y_synth]:
    df[label_col] = df[label_col].map(label_map)

y_train.to_csv(DATASET_DIR / "y_train.csv", index=False)
y_test.to_csv(DATASET_DIR / "y_test.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Label reindexing complete")

from katabatic.evaluate.tstr.evaluation import TSTREvaluation

print("Running TSTR evaluation")

tstr_start = time.time()

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(DATASET_DIR)
)

results = tstr.evaluate()

tstr_end = time.time()
print(f"TSTR completed in {(tstr_end - tstr_start)/60:.2f} minutes")

print("TSTR Results")
print(results)

pipeline_end = time.time()
print(f"Total pipeline runtime: {(pipeline_end - pipeline_start)/60:.2f} minutes")

print("SYNTHPOP (MAGIC) PIPELINE FINISHED SUCCESSFULLY")


ROOT: /content/drive/MyDrive/katabatic1

▶ Splitting MAGIC dataset
Loaded data with shape: (19020, 11)
Saved train/test full data
Train size: (15216, 11), Test size: (3804, 11)
Train label distribution:
 class
g    0.648396
h    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
g    0.648265
h    0.351735
Name: proportion, dtype: float64
Saved X/y split
Training shape: (15216, 10) (15216,)
Test shape: (3804, 10) (3804,)
✔ Split complete

▶ Running SynthPop (MAGIC)
✔ SynthPop generation complete
✔ Synthetic X / y saved

▶ Encoding features for TSTR
✔ Feature encoding complete

▶ Reindexing labels
Label mapping: {'g': 0, 'h': 1}
✔ Label reindexing complete

▶ Running TSTR evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/magic/synthpop_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.8265
F1 Score: 0.8230
AUC: 0.8693

MLP:
Accuracy: 0.8525
F1 Score: 0.8490
AUC: 0.9089

RF:
Accuracy: 0.8328
F1 Score: 0.8312
AUC: 0.8780

XGBoost:
Accuracy: 0.8423
F1 Score: 0.8427
AUC: 0.9012
✔ TSTR completed in 0.19 minutes

📊 TSTR RESULTS
{'LR': {'Accuracy': 0.8264984227129337, 'F1 Score': 0.8230009347108553, 'AUC': np.float64(0.8693471875200787)}, 'MLP': {'Accuracy': 0.8525236593059937, 'F1 Score': 0.8489717470801368, 'AUC': np.float64(0.9088876280948553)}, 'RF': {'Accuracy': 0.832807570977918, 'F1 Score': 0.8311887451053426, 'AUC': np.float64(0.8780312095015379)}, 'XGBoost': {'Accuracy': 0.8422712933753943, 'F1 Score': 0.8427382803711921, 'AUC': np.float64(0.9012252735862437)}}

⏱️ Total pipeline runtime: 0.38 minutes

✅ SYNTHPOP (MAGIC) PIPELINE FINISHED SUCCESSFULLY


In [ ]:
from pathlib import Path
import pandas as pd
import time
import subprocess

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "nursery.csv"
DATASET_DIR = ROOT / "sample_data" / "nursery"
SYNTH_DIR = ROOT / "synthetic" / "nursery" / "synthpop"

DATASET_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
pipeline_start = time.time()

from katabatic.utils.split_dataset import split_dataset

print("Splitting NURSERY dataset")

split_dataset(
    input_csv=str(RAW_CSV),
    output_dir=str(DATASET_DIR),
    label_col="8",
    test_size=0.2,
    stratify=True,
    random_state=42
)

print("Split complete")

print("Running SynthPop (NURSERY)")

x_train = pd.read_csv(DATASET_DIR / "x_train.csv")
y_train = pd.read_csv(DATASET_DIR / "y_train.csv")

label_col = "8"

train_full = pd.concat([x_train, y_train], axis=1)
train_csv = SYNTH_DIR / "train_full.csv"
train_full.to_csv(train_csv, index=False)

r_script = f"""
suppressMessages(library(synthpop))
set.seed(42)

data <- read.csv("{train_csv.as_posix()}")

syn_data <- syn(
  data,
  method = "cart",
  seed = 42
)

write.csv(
  syn_data$syn,
  "{(SYNTH_DIR / 'synthetic_full.csv').as_posix()}",
  row.names = FALSE
)
"""

r_path = SYNTH_DIR / "run_synthpop.R"
r_path.write_text(r_script.strip())

subprocess.run(
    ["Rscript", str(r_path)],
    check=True
)

print("SynthPop generation complete")

synth_full = pd.read_csv(SYNTH_DIR / "synthetic_full.csv")

def normalize_columns(df):
    df.columns = (
        df.columns
        .str.replace(".", "-", regex=False)
        .str.strip()
    )
    return df

synth_full = normalize_columns(synth_full)

if "8" in synth_full.columns:
    label_col_synth = "8"
elif "X8" in synth_full.columns:
    label_col_synth = "X8"
else:
    raise ValueError(f"Label column not found in synthetic data: {synth_full.columns}")

x_synth = synth_full.drop(columns=[label_col_synth])
y_synth = synth_full[[label_col_synth]]

x_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print(f"Synthetic X and y saved (label column = {label_col_synth})")

from sklearn.preprocessing import OrdinalEncoder

print("Encoding features for TSTR (column-aligned)")

x_test = pd.read_csv(DATASET_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

x_synth.columns = x_test.columns

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_test_enc = encoder.fit_transform(x_test.astype(str))
x_synth_enc = encoder.transform(x_synth.astype(str))

pd.DataFrame(x_test_enc, columns=x_test.columns).to_csv(
    DATASET_DIR / "x_test.csv", index=False
)

pd.DataFrame(x_synth_enc, columns=x_test.columns).to_csv(
    SYNTH_DIR / "x_synth.csv", index=False
)

print("Feature encoding complete (columns aligned)")

print("Reindexing labels")

y_train = pd.read_csv(DATASET_DIR / "y_train.csv")
y_test = pd.read_csv(DATASET_DIR / "y_test.csv")
y_synth = pd.read_csv(SYNTH_DIR / "y_synth.csv")

if "X8" in y_synth.columns:
    y_synth = y_synth.rename(columns={"X8": "8"})

label_col = "8"

for df in [y_train, y_test, y_synth]:
    df[label_col] = df[label_col].astype(str)

all_labels = pd.concat(
    [y_train[label_col], y_test[label_col], y_synth[label_col]]
).unique()

all_labels = sorted(all_labels)

label_map = {old: new for new, old in enumerate(all_labels)}
print("Label mapping:", label_map)

for df in [y_train, y_test, y_synth]:
    df[label_col] = df[label_col].map(label_map)

y_train.to_csv(DATASET_DIR / "y_train.csv", index=False)
y_test.to_csv(DATASET_DIR / "y_test.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Label reindexing complete")

from katabatic.evaluate.tstr.evaluation import TSTREvaluation

print("Running TSTR evaluation")

tstr_start = time.time()

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(DATASET_DIR)
)

results = tstr.evaluate()

tstr_end = time.time()
print(f"TSTR completed in {(tstr_end - tstr_start)/60:.2f} minutes")

print("TSTR Results")
print(results)

pipeline_end = time.time()
print(f"Total pipeline runtime: {(pipeline_end - pipeline_start)/60:.2f} minutes")

print("SYNTHPOP (NURSERY) PIPELINE FINISHED SUCCESSFULLY")


ROOT: /content/drive/MyDrive/katabatic1

▶ Splitting NURSERY dataset
Loaded data with shape: (12960, 9)
Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
not_recom     0.333333
priority      0.329186
spec_prior    0.312018
very_recom    0.025270
recommend     0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
not_recom     0.333333
priority      0.329090
spec_prior    0.312114
very_recom    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)
✔ Split complete

▶ Running SynthPop (NURSERY)
✔ SynthPop generation complete
✔ Synthetic X / y saved (label column = X8)

▶ Encoding features for TSTR (column-aligned)
✔ Feature encoding complete (columns aligned)

▶ Reindexing labels
Label mapping: {'not_recom': 0, 'priority': 1, 'recommend': 2, 'spec_prior': 3, 'very_recom': 4}
✔ Label reindexing complete

▶ Running TSTR evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [01:40:00] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/nursery/synthpop_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7623
F1 Score: 0.7551

MLP:
Accuracy: 0.9757
F1 Score: 0.9755

RF:
Accuracy: 0.9672
F1 Score: 0.9670

XGBoost:
Accuracy: 0.9765
F1 Score: 0.9764
✔ TSTR completed in 0.14 minutes

📊 TSTR RESULTS
{'LR': {'Accuracy': 0.7623456790123457, 'F1 Score': 0.7551233190065654}, 'MLP': {'Accuracy': 0.9756944444444444, 'F1 Score': 0.9754741814050685}, 'RF': {'Accuracy': 0.9672067901234568, 'F1 Score': 0.9670168902813339}, 'XGBoost': {'Accuracy': 0.9764660493827161, 'F1 Score': 0.9763707794429647}}

⏱️ Total pipeline runtime: 0.25 minutes

✅ SYNTHPOP (NURSERY) PIPELINE FINISHED SUCCESSFULLY


In [ ]:
from pathlib import Path
import pandas as pd
import time
import subprocess

ROOT = Path("/content/drive/MyDrive/katabatic1")

RAW_CSV = ROOT / "raw_data" / "car.csv"
DATASET_DIR = ROOT / "sample_data" / "car"
SYNTH_DIR = ROOT / "synthetic" / "car" / "synthpop"

DATASET_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
pipeline_start = time.time()

from katabatic.utils.split_dataset import split_dataset

print("Splitting CAR dataset")

split_dataset(
    input_csv=str(RAW_CSV),
    output_dir=str(DATASET_DIR),
    label_col="6",
    test_size=0.2,
    stratify=True,
    random_state=42
)

print("Split complete")

print("Running SynthPop (CAR)")

x_train = pd.read_csv(DATASET_DIR / "x_train.csv")
y_train = pd.read_csv(DATASET_DIR / "y_train.csv")

label_col = "6"

train_full = pd.concat([x_train, y_train], axis=1)
train_csv = SYNTH_DIR / "train_full.csv"
train_full.to_csv(train_csv, index=False)

r_script = f"""
suppressMessages(library(synthpop))
set.seed(42)

data <- read.csv("{train_csv.as_posix()}")

syn_data <- syn(
  data,
  method = "cart",
  seed = 42
)

write.csv(
  syn_data$syn,
  "{(SYNTH_DIR / 'synthetic_full.csv').as_posix()}",
  row.names = FALSE
)
"""

r_path = SYNTH_DIR / "run_synthpop.R"
r_path.write_text(r_script.strip())

subprocess.run(
    ["Rscript", str(r_path)],
    check=True
)

print("SynthPop generation complete")

synth_full = pd.read_csv(SYNTH_DIR / "synthetic_full.csv")

if "6" in synth_full.columns:
    label_col_synth = "6"
elif "X6" in synth_full.columns:
    label_col_synth = "X6"
else:
    raise ValueError(f"Label column not found: {synth_full.columns}")

x_synth = synth_full.drop(columns=[label_col_synth])
y_synth = synth_full[[label_col_synth]]

x_synth.to_csv(SYNTH_DIR / "x_synth.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print(f"Synthetic X and y saved (label column = {label_col_synth})")

from sklearn.preprocessing import OrdinalEncoder

print("Encoding features for TSTR (column-aligned)")

x_test = pd.read_csv(DATASET_DIR / "x_test.csv")
x_synth = pd.read_csv(SYNTH_DIR / "x_synth.csv")

x_synth.columns = x_test.columns

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_test_enc = encoder.fit_transform(x_test.astype(str))
x_synth_enc = encoder.transform(x_synth.astype(str))

pd.DataFrame(x_test_enc, columns=x_test.columns).to_csv(
    DATASET_DIR / "x_test.csv", index=False
)

pd.DataFrame(x_synth_enc, columns=x_test.columns).to_csv(
    SYNTH_DIR / "x_synth.csv", index=False
)

print("Feature encoding complete (columns aligned)")

print("Reindexing labels")

y_train = pd.read_csv(DATASET_DIR / "y_train.csv")
y_test = pd.read_csv(DATASET_DIR / "y_test.csv")
y_synth = pd.read_csv(SYNTH_DIR / "y_synth.csv")

if "X6" in y_synth.columns:
    y_synth = y_synth.rename(columns={"X6": "6"})

label_col = "6"

for df in [y_train, y_test, y_synth]:
    df[label_col] = df[label_col].astype(str)

all_labels = pd.concat(
    [y_train[label_col], y_test[label_col], y_synth[label_col]]
).unique()

all_labels = sorted(all_labels)

label_map = {old: new for new, old in enumerate(all_labels)}
print("Label mapping:", label_map)

for df in [y_train, y_test, y_synth]:
    df[label_col] = df[label_col].map(label_map)

y_train.to_csv(DATASET_DIR / "y_train.csv", index=False)
y_test.to_csv(DATASET_DIR / "y_test.csv", index=False)
y_synth.to_csv(SYNTH_DIR / "y_synth.csv", index=False)

print("Label reindexing complete")

from katabatic.evaluate.tstr.evaluation import TSTREvaluation

print("Running TSTR evaluation")

tstr_start = time.time()

tstr = TSTREvaluation(
    synthetic_dir=str(SYNTH_DIR),
    real_test_dir=str(DATASET_DIR)
)

results = tstr.evaluate()

tstr_end = time.time()
print(f"TSTR completed in {(tstr_end - tstr_start)/60:.2f} minutes")

print("TSTR Results")
print(results)

pipeline_end = time.time()
print(f"Total pipeline runtime: {(pipeline_end - pipeline_start)/60:.2f} minutes")

print("SYNTHPOP (CAR) PIPELINE FINISHED SUCCESSFULLY")


ROOT: /content/drive/MyDrive/katabatic1

▶ Splitting CAR dataset
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
unacc    0.700434
acc      0.222142
good     0.039797
vgood    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
unacc    0.699422
acc      0.222543
good     0.040462
vgood    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
✔ Split complete

▶ Running SynthPop (CAR)
✔ SynthPop generation complete
✔ Synthetic X / y saved (label column = X6)

▶ Encoding features for TSTR (column-aligned)
✔ Feature encoding complete (columns aligned)

▶ Reindexing labels
Label mapping: {'acc': 0, 'good': 1, 'unacc': 2, 'vgood': 3}
✔ Label reindexing complete

▶ Running TSTR evaluation


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [01:41:42] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results/car/synthpop_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6850
F1 Score: 0.6163

MLP:
Accuracy: 0.9393
F1 Score: 0.9365

RF:
Accuracy: 0.9220
F1 Score: 0.9215

XGBoost:
Accuracy: 0.9364
F1 Score: 0.9355
✔ TSTR completed in 0.03 minutes

📊 TSTR RESULTS
{'LR': {'Accuracy': 0.684971098265896, 'F1 Score': 0.6163172129872305}, 'MLP': {'Accuracy': 0.9393063583815029, 'F1 Score': 0.9364748046628626}, 'RF': {'Accuracy': 0.9219653179190751, 'F1 Score': 0.9215449457972705}, 'XGBoost': {'Accuracy': 0.9364161849710982, 'F1 Score': 0.9355182480050687}}

⏱️ Total pipeline runtime: 0.09 minutes

✅ SYNTHPOP (CAR) PIPELINE FINISHED SUCCESSFULLY
